In [17]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# TABLA ALEMANIA — Limpieza completa paso a paso
# ------------------------------------------------------------
# 🌍 Este dataset tiene información de todos los municipios de
#    Alemania: nombres, códigos postales, población, coordenadas...
#    Es como un directorio telefónico gigante del país, pero con
#    datos de cada pueblo y ciudad. Vamos a dejarlo limpio y listo
#    para trabajar, paso a paso y con semáforos para ver qué está bien.
print("-" * 60)
print("INICIO LIMPIEZA: germany_municipalities")
print("-" * 60)

# ------------------------------------------------------------
# 1. CARGA
# ------------------------------------------------------------
# 📂 Abrimos el CSV. Este archivo tiene un detalle: usa encoding
#    "utf-8-sig", que es como un UTF-8 pero con un "sello" al inicio
#    (BOM) que meten algunos programas de Windows. Si no lo usamos,
#    la primera columna aparecería con caracteres basura al inicio.
# Igual que antes: leemos todo como texto y no dejamos que Pandas
# invente nulos por su cuenta. Primero ver el archivo tal cual.
df_de = pd.read_csv(
    "germany_municipalities_jun_2025.csv",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)
print(f"\n[1] 📥 Cargado: {df_de.shape[0]} filas x {df_de.shape[1]} columnas")
print(f"    → Es como abrir un directorio telefónico de todo un país:")
print(f"      {df_de.shape[0]:,} municipios (filas), cada uno con {df_de.shape[1]} datos (columnas).")
print(f"    Columnas: {df_de.columns.tolist()}")

# - Guardamos los nombres originales para después poder auditar
#   qué se renombró y qué se eliminó. Es como guardar la foto
#   del "antes" para compararla con el "después" al final.
cols_originales = df_de.columns.tolist()

# ------------------------------------------------------------
# 2. RENAME
# ------------------------------------------------------------
# 🏷️ Los nombres de las columnas vienen con paréntesis y mezcla de
#    inglés/alemán: "municipality_name_(gemeindename)". Eso es como
#    una etiqueta escrita por dos personas distintas: funciona, pero
#    es incómoda. Las renombramos a algo más limpio y en un solo idioma.
df_de = df_de.rename(columns={
    "municipality_name_(gemeindename)": "municipality_name",
    "municipality_designation": "municipality_type",
    "regional_designation": "regional_type",
    "per_km2": "population_density",
    "state_(land)": "state",
    "district_(kreis)": "district",
})
print(f"\n[2] 🏷️ Rename hecho. Columnas nuevas: {df_de.columns.tolist()}")
print(f"    → 'municipality_name_(gemeindename)' ahora se llama 'municipality_name'.")
print(f"      Más corto, más claro, más fácil de escribir en el código.")

# ------------------------------------------------------------
# 2.5 DIAGNÓSTICO DE NULOS Y VACÍOS
# ------------------------------------------------------------
# 🩺 Antes de operar, diagnosticamos. Contamos cuántos vacíos ("")
#    y cuántos nulos reales (NaN) hay en cada columna, y lo mostramos
#    ordenado de la más problemática a la más limpia.
print("\n" + "-" * 60)
print("[2.5] 🩺 DIAGNÓSTICO DE NULOS Y VACÍOS")
print("-" * 60)

reporte_nulos = pd.DataFrame({
    "tipo_dato":       df_de.dtypes,
    "vacios_str":      (df_de == "").sum(),
    "nan_reales":      df_de.isnull().sum(),
    "total_problemas": ((df_de == "") | df_de.isnull()).sum(),
})
reporte_nulos["pct_del_total"] = (
    reporte_nulos["total_problemas"] / len(df_de) * 100
).round(2)
reporte_nulos = reporte_nulos.sort_values("total_problemas", ascending=False)
print(reporte_nulos.to_string())

# 🚦 Y ahora el semáforo: un color por columna según qué tan grave es.
#    Es como el tablero de un auto: verde = todo bien, rojo = peligro.
print("\nSemáforo por columna:")
for col in reporte_nulos.index:
    pct = reporte_nulos.loc[col, "pct_del_total"]
    if   pct == 0:   icono = "✅"  # Perfecto, sin problemas.
    elif pct < 5:    icono = "🟢"  # Casi limpio, muy poquitos vacíos.
    elif pct < 30:   icono = "🟡"  # Ojo, hay que prestarle atención.
    elif pct < 70:   icono = "🟠"  # Bastante sucio, hay que trabajarlo.
    else:            icono = "🔴"  # Casi vacío, cuidado con esta columna.
    print(f"  {icono}  {col:<25} → {pct:>6.2f}%  "
          f"({reporte_nulos.loc[col, 'total_problemas']} filas)")

# Ahora sí, convertimos los vacíos en nulos reales. Es como marcar
# con resaltador las casillas en blanco, para encontrarlas fácil.
df_de = df_de.replace({"": np.nan})
print("\n[2.5] 🔄 Vacíos convertidos a NaN reales.")

# ------------------------------------------------------------
# 2.75 ESTANDARIZAR TEXTO
# ------------------------------------------------------------
# 🔤 Los textos (estados, distritos, nombres de municipios) vienen
#    con mayúsculas, minúsculas, espacios extra y hasta comillas.
#    Es como si cada quien hubiera escrito las etiquetas a su manera.
#    Los unificamos para poder agrupar y comparar sin sorpresas.
print("\n" + "-" * 60)
print("[2.75] 🔤 ESTANDARIZACIÓN DE TEXTO")
print("-" * 60)

text_cols = [
    "state",
    "district",
    "regional_type",
    "municipality_name",
    "municipality_type",
]

# Primero nos aseguramos de que sean texto puro (por si algún número
# se coló como texto en algún lado). Es como asegurarnos de tener
# todas las piezas del mismo material antes de pintarlas.
for col in text_cols:
    df_de[col] = df_de[col].astype(str)

# ✂️ A minúsculas y sin espacios en las orillas: "  Bavaria " → "bavaria".
for col in text_cols:
    df_de[col] = df_de[col].str.lower().str.strip()

# 🧹 Quitamos las comillas de las orillas: '"berlin"' → 'berlin'.
#    Es como pelar una naranja antes de comérsela.
for col in text_cols:
    df_de[col] = df_de[col].str.strip('"')

# 🧽 Colapsamos espacios dobles: "baden   wurttemberg" → "baden wurttemberg".
for col in text_cols:
    df_de[col] = df_de[col].str.replace(r"\s+", " ", regex=True)

# Cuando convertimos a str, los NaN se volvieron el texto "nan".
# Aquí los regresamos a nulos reales. Es como deshacer un disfraz.
df_de[text_cols] = df_de[text_cols].replace({"nan": np.nan, "": np.nan})

print("Estandarización aplicada a:")
for col in text_cols:
    print(f"  · {col}")

print(f"\nEjemplo 'state' únicos: {df_de['state'].dropna().unique()[:5]}")
print(f"Ejemplo 'district' únicos: {df_de['district'].dropna().unique()[:5]}")
print(f"  → Antes había 'Bavaria', 'BAVARIA', '  bavaria'; ahora todos son 'bavaria'.")

# ------------------------------------------------------------
# 3. NULOS LÓGICOS / VÁLIDOS
# ------------------------------------------------------------
# 🤔 Aquí un matiz importante: no todos los nulos son errores.
#    Un municipio sin "tipo" probablemente ES un municipio normal,
#    solo que nadie lo clasificó explícitamente. Igual con el
#    "regional_type": si no tiene designación, es que no aplica.
antes_nulos_mt = df_de["municipality_type"].isnull().sum()
antes_nulos_rt = df_de["regional_type"].isnull().sum()

df_de["municipality_type"] = df_de["municipality_type"].fillna("Municipio")
df_de["regional_type"]     = df_de["regional_type"].fillna("Sin Designación")

print(f"\n[3] 🤔 Nulos lógicos rellenados:")
print(f"    municipality_type: {antes_nulos_mt} filas → 'Municipio'")
print(f"    regional_type:     {antes_nulos_rt} filas → 'Sin Designación'")
print(f"  → Es como cuando en un formulario dejas 'tipo de documento' en blanco:")
print(f"    se asume que es el más común. Aquí, el tipo más común es 'Municipio'.")

# ------------------------------------------------------------
# 4. BUGS CRÍTICOS → dropna
# ------------------------------------------------------------
# 🫀 Hay datos que son como el "corazón" de una fila: si faltan,
#    la fila pierde sentido. Sin código postal o sin coordenadas,
#    un municipio no nos sirve para nada geográfico. Los eliminamos.
antes = len(df_de)
criticas = ["postal_code", "longitude", "latitude"]
print(f"\n[4] 🫀 Nulos en columnas críticas antes del dropna:")
print(df_de[criticas].isnull().sum().to_string())

df_de = df_de.dropna(subset=criticas)
print(f"    📉 Filas eliminadas: {antes - len(df_de)}")
print(f"    ✅ Filas restantes:  {len(df_de)}")
print(f"  → Es como tirar direcciones incompletas: si no hay código postal")
print(f"    ni coordenadas, el municipio no se puede ubicar en el mapa.")

# ------------------------------------------------------------
# 5. PARSEAR COORDENADAS
# ------------------------------------------------------------
# 🗺️ Las coordenadas vienen como texto, a veces con coma decimal
#    europea ("13,405" en vez de "13.405") y a veces con valores
#    rotos como "1340500" (un decimal perdido). Nuestra función
#    limpia todo eso: convierte comas en puntos y, si el número
#    es demasiado grande para ser una coordenada real, lo divide
#    entre 10 hasta que quede en el rango válido.
def parse_coord(value, tipo):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    s = str(value).strip().replace(",", ".")
    try:
        num = float(s)
    except ValueError:
        return np.nan
    # Si un decimal se perdió al exportar, el número queda enorme.
    # Dividir entre 10 hasta que encaje en el rango correcto
    # es como "bajarle el volumen" al número hasta que suene bien.
    if tipo == "longitude":
        while num > 180:
            num /= 10
    else:
        while num > 90:
            num /= 10
    return num

df_de["longitude"] = df_de["longitude"].apply(lambda v: parse_coord(v, "longitude"))
df_de["latitude"]  = df_de["latitude"].apply(lambda v: parse_coord(v, "latitude"))

coords_rotas = df_de["longitude"].isnull().sum() + df_de["latitude"].isnull().sum()
print(f"\n[5] 🗺️ Coordenadas parseadas")
print(f"    Longitud  → min: {df_de['longitude'].min():.4f} | max: {df_de['longitude'].max():.4f}")
print(f"    Latitud   → min: {df_de['latitude'].min():.4f} | max: {df_de['latitude'].max():.4f}")
print(f"    ⚠️  Coords que no se pudieron parsear: {coords_rotas}")
print(f"  → Alemania está más o menos entre longitud 5.5 y 15.5, y latitud")
print(f"    47 y 55.5. Si los min/max caen dentro de esos rangos, todo bien.")

# ------------------------------------------------------------
# 6. CONVERTIR NUMÉRICOS
# ------------------------------------------------------------
# 🔢 Estas columnas deben ser números, pero llegaron como texto
#    porque al inicio le dijimos a Pandas que leyera todo como str.
#    Ahora las convertimos a número. Si alguna celda tiene texto
#    basura que no se puede convertir, queda como nulo (coerce).
for col in ["area_km2", "population_total", "male", "female"]:
    antes_nulos = df_de[col].isnull().sum()
    df_de[col] = pd.to_numeric(df_de[col], errors="coerce")
    despues_nulos = df_de[col].isnull().sum()
    print(f"\n[6] 🔢 {col}: {antes_nulos} → {despues_nulos} nulos tras to_numeric")
print(f"  → Si algún nulo apareció de la nada, es porque esa celda tenía")
print(f"    texto que no se podía leer como número. Mejor perder ese dato")
print(f"    que arrastrar un valor mal calculado.")

# ------------------------------------------------------------
# 7. POBLACIÓN según tipo de municipio
# ------------------------------------------------------------
# 💡 Hay municipios especiales en Alemania llamados "gemeindefreies
#    Gebiet" (área libre de municipio): son zonas sin población real,
#    como bosques o terrenos militares. Para esos, la población
#    nula ES correcta: debería ser 0. No son datos faltantes.
mask_no_habitado = df_de["municipality_type"].str.contains("gemfr", case=False, na=False)
print(f"\n[7] 💡 Municipios 'gemfr. gebiet' (sin población real): {mask_no_habitado.sum()}")
print(f"  → Son áreas deshabitadas por definición, no errores del sistema.")

# Solo miramos los habitados: esos sí deberían tener población.
mask_habitado = ~mask_no_habitado
poblacion_nula_habitado = df_de.loc[mask_habitado, "population_total"].isnull().sum()
print(f"    Municipios habitados con población nula: {poblacion_nula_habitado}")

# Rellenamos con 0 solo los no habitados (correcto).
df_de.loc[mask_no_habitado, "population_total"] = (
    df_de.loc[mask_no_habitado, "population_total"].fillna(0)
)
# Y eliminamos los habitados con población nula: ahí sí hay un error.
antes = len(df_de)
df_de = df_de.dropna(subset=["population_total"])
print(f"    📉 Filas eliminadas por población nula: {antes - len(df_de)}")

# ------------------------------------------------------------
# 8. DENSIDAD recalculada + drop de area_km2
# ------------------------------------------------------------
# 🧮 La columna "per_km2" (densidad original) puede venir mal calculada
#    o traer valores raros. Mejor la recalculamos nosotros mismos a
#    partir de población y área. Es como verificar una cuenta ajena
#    con calculadora propia.
#
# ⚠️ Ojo con la división por cero: si algún municipio tiene área 0
#    (probablemente un error), la división daría infinito. Lo tratamos.
area_cero = (df_de["area_km2"] == 0).sum()
print(f"\n[8] 🧮 Densidad recalculada:")
print(f"    Filas con area_km2 = 0 (dividirían por 0): {area_cero}")

df_de["population_density"] = df_de["population_total"] / df_de["area_km2"]
# Los infinitos se convierten en nulos: mejor un hueco que un valor absurdo.
df_de["population_density"] = df_de["population_density"].replace(
    [np.inf, -np.inf], np.nan
)

# Después de recalcular la densidad, ya no necesitamos area_km2.
# Es como usar una escalera para subir al techo y luego tirarla:
# ya cumplió su función, no hay por qué cargarla.
df_de = df_de.drop(columns=["area_km2"])
print(f"    ✂️ Columna 'area_km2' eliminada.")
print(f"  → La usamos para calcular la densidad, y ahora ya no la necesitamos.")

# ------------------------------------------------------------
# 9. AUDITORÍA FINAL — comparativa antes/después
# ------------------------------------------------------------
# 🧾 Repaso final antes de guardar. Como el chef que prueba el plato
#    antes de sacarlo al cliente: verificamos que todo esté en orden.
print("\n" + "-" * 60)
print("🧾 AUDITORÍA FINAL")
print("-" * 60)
print(f"Filas finales: {len(df_de):,}")
print(f"\n🔍 Nulos restantes por columna:")
print(df_de.isnull().sum().sort_values(ascending=False))
print(f"\n🔑 Duplicados: {df_de.duplicated().sum()}")
print(f"  → Si es 0, cada fila es única. Si es >0, hay que investigar.")
print(f"\n👀 Primeras 3 filas:")
print(df_de.head(3).to_string())
print(f"\n📋 Tipos de datos:")
print(df_de.dtypes)

# ------------------------------------------------------------
# 10. GUARDAR
# ------------------------------------------------------------
# 💾 Guardamos todo limpio en un CSV nuevo. Es como sacar una foto
#    del trabajo terminado para no repetir la limpieza después.
df_de.to_csv("germany_municipalities_clean.csv", index=False)
print(f"\n✅ Guardado: germany_municipalities_clean.csv ({len(df_de)} filas)")

# ------------------------------------------------------------
# 11. VALIDACIONES POST-GUARDADO
# ------------------------------------------------------------
# 📸 Y ahora la parte más importante: volvemos a abrir el archivo
#    que acabamos de guardar y lo revisamos todo de nuevo.
#    Es como mirar la foto que acabamos de tomar por si salió borrosa.
print("\n" + "=" * 60)
print("🔍 VALIDACIONES POST-GUARDADO")
print("=" * 60)

# - Recargar el CSV guardado para verificar integridad
df_check = pd.read_csv("germany_municipalities_clean.csv")

# ------------------------------------------------------------
# 11.1 Resumen general
# ------------------------------------------------------------
print(f"\n[11.1] 📊 RESUMEN GENERAL")
print(f"    Filas guardadas:    {df_check.shape[0]:,}")
print(f"    Columnas guardadas: {df_check.shape[1]}")
print(f"    Tamaño en memoria:  {df_check.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"  → Si las filas coinciden con las que teníamos antes de guardar,")
print(f"    el archivo se escribió bien. Si no, algo se perdió en el camino.")

# ------------------------------------------------------------
# 11.2 Columnas finales (orden exacto)
# ------------------------------------------------------------
print(f"\n[11.2] 📜 COLUMNAS FINALES ({df_check.shape[1]}):")
for i, col in enumerate(df_check.columns, 1):
    print(f"    {i:>2}. {col:<25} ({df_check[col].dtype})")
print(f"  → Este es el inventario exacto de lo que hay en la bodega final.")

# ------------------------------------------------------------
# 11.3 Columnas eliminadas
# ------------------------------------------------------------
print(f"\n[11.3] ✂️ COLUMNAS ELIMINADAS:")
# Aquí comparamos lo original con lo final, ajustando por los renombres:
# una columna renombrada no es "eliminada", solo cambió de nombre.
cols_renombradas = {
    "municipality_name_(gemeindename)": "municipality_name",
    "municipality_designation": "municipality_type",
    "regional_designation": "regional_type",
    "per_km2": "population_density",
    "state_(land)": "state",
    "district_(kreis)": "district",
}

# - Ajustando las que fueron renombradas (no son "eliminadas")
cols_originales_efectivas = [
    cols_renombradas.get(c, c) for c in cols_originales
]
cols_eliminadas = [
    c for c in cols_originales_efectivas if c not in df_check.columns
]

if cols_eliminadas:
    for col in cols_eliminadas:
        print(f"    ✗ {col}")
else:
    print("    (ninguna)")

print(f"\n    Total columnas originales:  {len(cols_originales)}")
print(f"    Total columnas renombradas: {len(cols_renombradas)}")
print(f"    Total columnas eliminadas:  {len(cols_eliminadas)}")
print(f"    Total columnas finales:     {len(df_check.columns)}")
print(f"  → Si renombradas + eliminadas + conservadas = originales,")
print(f"    no se nos escapó ninguna columna por el camino.")

# ------------------------------------------------------------
# 11.4 Verificación de nulos restantes
# ------------------------------------------------------------
print(f"\n[11.4] 🔍 NULOS EN EL CSV GUARDADO:")
nulos_check = df_check.isnull().sum()
nulos_check = nulos_check[nulos_check > 0].sort_values(ascending=False)

if len(nulos_check) == 0:
    print("    ✅ Sin nulos restantes")
    print("    → La tabla quedó limpia como un vidrio recién lavado.")
else:
    for col, n in nulos_check.items():
        pct = n / len(df_check) * 100
        # 🚦 Semáforo también aquí, para ver rápido qué columnas
        #    conservan huecos y cuánta gravedad tiene.
        if   pct < 5:    icono = "🟢"
        elif pct < 30:   icono = "🟡"
        elif pct < 70:   icono = "🟠"
        else:            icono = "🔴"
        print(f"    {icono} {col:<25} → {n:>6} filas ({pct:>6.2f}%)")

# ------------------------------------------------------------
# 11.5 Verificación de duplicados
# ------------------------------------------------------------
dups = df_check.duplicated().sum()
print(f"\n[11.5] 🔑 DUPLICADOS EN EL CSV GUARDADO: {dups}")
if dups == 0:
    print("    ✅ Sin duplicados")
    print("    → Cada fila es única, no hay municipios repetidos.")
else:
    print(f"    ⚠️  {dups} filas duplicadas")
    print(f"    → Habría que investigar por qué se repitieron.")

# ------------------------------------------------------------
# 11.6 Verificación de columnas que NO deben estar
# ------------------------------------------------------------
print(f"\n[11.6] 🚫 VERIFICACIÓN DE COLUMNAS ELIMINADAS:")
# Esta verificación es como revisar que la basura que sacamos
# efectivamente NO se quedó dentro de la casa.
cols_prohibidas = ["area_km2", "per_km2", "municipality_name_(gemeindename)",
                   "municipality_designation", "regional_designation",
                   "state_(land)", "district_(kreis)"]
encontradas = [c for c in cols_prohibidas if c in df_check.columns]
if encontradas:
    print(f"    ⚠️  Columnas que NO deberían estar: {encontradas}")
else:
    print("    ✅ Ninguna columna eliminada/renombrada quedó en el CSV")
    print("    → La limpieza fue efectiva: lo que sacamos, se quedó fuera.")

# ------------------------------------------------------------
# 11.7 Verificación de tipos de datos
# ------------------------------------------------------------
print(f"\n[11.7] 📋 TIPOS DE DATOS:")
print(df_check.dtypes.to_string())
print(f"  → Los tipos numéricos deberían ser float/int, y los textos object.")
print(f"    Si algún número aparece como object, hay texto escondido adentro.")

# ------------------------------------------------------------
# 11.8 Verificación de rangos de coordenadas
# ------------------------------------------------------------
# 🗺️ Verificación geográfica: si las coordenadas caen todas dentro
#    del "rectángulo" de Alemania, todo tiene sentido. Si alguna
#    cae en el océano o en otro país, algo se rompió al parsear.
print(f"\n[11.8] 🗺️ RANGOS DE COORDENADAS:")
if "longitude" in df_check.columns and "latitude" in df_check.columns:
    print(f"    Longitud  → min: {df_check['longitude'].min():>8.4f} | "
          f"max: {df_check['longitude'].max():>8.4f}")
    print(f"    Latitud   → min: {df_check['latitude'].min():>8.4f} | "
          f"max: {df_check['latitude'].max():>8.4f}")

    # - Validar rango geográfico de Alemania (aprox.)
    lon_ok = df_check["longitude"].between(5.5, 15.5).all()
    lat_ok = df_check["latitude"].between(47.0, 55.5).all()
    print(f"    ¿Longitudes dentro de Alemania (5.5–15.5)?:  {lon_ok}")
    print(f"    ¿Latitudes dentro de Alemania (47.0–55.5)?: {lat_ok}")
    if lon_ok and lat_ok:
        print(f"  → ✅ Todas las coordenadas caen dentro de Alemania.")
        print(f"    Ningún municipio apareció flotando en el Atlántico.")
    else:
        print(f"  → ⚠️  Hay coordenadas fuera de Alemania: hay que revisar el parseo.")
else:
    print("    ⚠️  Columnas de coordenadas no encontradas")

# ------------------------------------------------------------
# 11.9 Verificación de densidad
# ------------------------------------------------------------
# 📊 La densidad de población debería ser un número razonable:
#    ningún municipio tiene millones de personas por km², ni
#    debería haber densidades negativas (sería absurdo).
print(f"\n[11.9] 📊 VERIFICACIÓN DE DENSIDAD:")
if "population_density" in df_check.columns:
    print(f"    Densidad → min: {df_check['population_density'].min():>10.2f} | "
          f"max: {df_check['population_density'].max():>10.2f}")
    print(f"    Densidad → media: {df_check['population_density'].mean():>10.2f}")
    nulos_dens = df_check["population_density"].isnull().sum()
    print(f"    Densidad nula: {nulos_dens} filas")
    if nulos_dens > 0:
        print(f"    → Esos son los municipios con area_km2 = 0 (división por cero).")

# ------------------------------------------------------------
# 11.10 Verificación de población
# ------------------------------------------------------------
# 🇩🇪 La población total de Alemania ronda los 83 millones de personas.
#    Si la suma se aleja mucho de ese número, algo se rompió al
#    convertir los datos. También verificamos que male + female
#    sea aproximadamente igual a population_total.
print(f"\n[11.10] 🇩🇪 VERIFICACIÓN DE POBLACIÓN:")
if "population_total" in df_check.columns:
    print(f"    Población total del país: {df_check['population_total'].sum():,.0f}")
    print(f"    → Alemania tiene ~83 millones de habitantes.")
    print(f"    Población → min: {df_check['population_total'].min():>10.0f} | "
          f"max: {df_check['population_total'].max():>10.0f}")
    if "male" in df_check.columns and "female" in df_check.columns:
        suma_mf = df_check["male"].sum() + df_check["female"].sum()
        diff = abs(suma_mf - df_check["population_total"].sum())
        print(f"    Suma male + female: {suma_mf:,.0f}")
        print(f"    Diferencia vs population_total: {diff:,.0f}")
        print(f"    → Si la diferencia es pequeña, los subtotales cuadran")
        print(f"      con el total. Si es enorme, hay un error de captura.")

# ------------------------------------------------------------
# 11.11 Verificación de estados únicos
# ------------------------------------------------------------
# 🗾 Alemania tiene 16 estados federados (Bundesländer). Si después
#    de limpiar hay 16 estados únicos, todo cuadra. Si hay más, es
#    porque se colaron variantes mal escritas.
print(f"\n[11.11] 🗾 ESTADOS ÚNICOS (state):")
if "state" in df_check.columns:
    estados = df_check["state"].dropna().unique()
    print(f"    Total estados: {len(estados)}")
    print(f"    → Alemania tiene 16 estados federados. Si aquí aparecen más,")
    print(f"      es que hay duplicados con ortografía distinta.")
    for e in sorted(estados):
        print(f"      · {e}")

# ------------------------------------------------------------
# 11.12 Resultado final
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("✅ VALIDACIONES COMPLETADAS")
print("=" * 60)
print(f"📁 Archivo: germany_municipalities_clean.csv")
print(f"📊 Filas:   {df_check.shape[0]:,}")
print(f"📋 Columnas: {df_check.shape[1]}")
print(f"✂️  Eliminadas: {len(cols_eliminadas)} columnas")
print("=" * 60)
print(f"→ Con esto, la tabla de municipios alemanes quedó lista")
print(f"  para el análisis: sin coordenadas rotas, sin textos repetidos,")
print(f"  y con la densidad recalculada en base a datos confiables.")

------------------------------------------------------------
INICIO LIMPIEZA: germany_municipalities
------------------------------------------------------------

[1] 📥 Cargado: 10950 filas x 13 columnas
    → Es como abrir un directorio telefónico de todo un país:
      10,950 municipios (filas), cada uno con 13 datos (columnas).
    Columnas: ['state_(land)', 'district_(kreis)', 'regional_designation', 'municipality_name_(gemeindename)', 'municipality_designation', 'area_km2', 'population_total', 'male', 'female', 'per_km2', 'postal_code', 'longitude', 'latitude']

[2] 🏷️ Rename hecho. Columnas nuevas: ['state', 'district', 'regional_type', 'municipality_name', 'municipality_type', 'area_km2', 'population_total', 'male', 'female', 'population_density', 'postal_code', 'longitude', 'latitude']
    → 'municipality_name_(gemeindename)' ahora se llama 'municipality_name'.
      Más corto, más claro, más fácil de escribir en el código.

-----------------------------------------------------

In [18]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# TABLA SEGUROS — Limpieza completa (v4, sin nulos finales)
# ------------------------------------------------------------
print("-" * 60)
print("INICIO LIMPIEZA: insurance_policies_claims")
print("-" * 60)

# ------------------------------------------------------------
# 1. CARGA
# ------------------------------------------------------------
df_in = pd.read_csv(
    "insurance_policies_claims_2018_2025.csv",
    dtype=str,
    keep_default_na=False,
)
print(f"\n[1] 📥 Cargado: {df_in.shape[0]} filas x {df_in.shape[1]} columnas")

# ------------------------------------------------------------
# 1.5 CARGAR TABLA ALEMANIA
# ------------------------------------------------------------
df_germany = pd.read_csv(
    "germany_municipalities_jun_2025.csv",
    dtype=str,
    keep_default_na=False,
)
print(f"    Alemania cargada: {df_germany.shape[0]} filas")

# ------------------------------------------------------------
# 2. RENAME
# ------------------------------------------------------------
print(f"\n[2] 🏷️ Columnas ya estandarizadas (no se requiere rename)")

# ------------------------------------------------------------
# 2.1 DROP customer_email
# ------------------------------------------------------------
cols_a_eliminar = ["customer_email"]
cols_presentes = [c for c in cols_a_eliminar if c in df_in.columns]
if cols_presentes:
    df_in = df_in.drop(columns=cols_presentes)
    print(f"\n[2.1] ✂️ Columnas eliminadas: {cols_presentes}")

# ------------------------------------------------------------
# 2.5 DIAGNÓSTICO DE NULOS
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[2.5] 🩺 DIAGNÓSTICO DE NULOS Y VACÍOS")
print("-" * 60)

reporte_nulos = pd.DataFrame({
    "tipo_dato":       df_in.dtypes,
    "vacios_str":      (df_in == "").sum(),
    "nan_reales":      df_in.isnull().sum(),
    "total_problemas": ((df_in == "") | df_in.isnull()).sum(),
})
reporte_nulos["pct_del_total"] = (reporte_nulos["total_problemas"] / len(df_in) * 100).round(2)
reporte_nulos = reporte_nulos.sort_values("total_problemas", ascending=False)
print(reporte_nulos[reporte_nulos["total_problemas"] > 0].to_string())

df_in = df_in.replace({"": np.nan})

# ------------------------------------------------------------
# 2.75 ESTANDARIZAR TEXTO
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[2.75] 🔤 ESTANDARIZACIÓN DE TEXTO")
print("-" * 60)

cat_cols = ["policy_status", "claim_status", "insurance_type", "coverage_type",
            "payment_frequency", "underwriting_decision"]
for col in cat_cols:
    df_in[col] = df_in[col].str.lower().str.strip()

id_cols = ["policy_id", "policy_number", "customer_id", "agent_id",
           "broker_id", "adjuster_id", "claim_id", "postal_code"]
for col in id_cols:
    if col in df_in.columns:
        df_in[col] = df_in[col].str.strip()

name_cols = ["customer_name", "agent_name", "adjuster_name", "address", "city"]
for col in name_cols:
    if col in df_in.columns:
        df_in[col] = df_in[col].str.strip().str.replace(r"\s+", " ", regex=True)

def normalizar_telefono(v):
    if pd.isna(v):
        return np.nan
    s = "".join(c for c in str(v).strip() if c.isdigit() or c == "+")
    return s if len(s) >= 8 else np.nan

if "customer_phone" in df_in.columns:
    df_in["customer_phone"] = df_in["customer_phone"].apply(normalizar_telefono)

# Género
def normalizar_genero(v):
    if pd.isna(v):
        return np.nan
    g = str(v).strip().lower()
    if g in ("m", "male"):
        return "M"
    if g in ("f", "female"):
        return "F"
    if g in ("other", "o", "divers", "diverse", "non-binary", "nonbinary"):
        return "Other"
    if g in ("no indica", "no indica.", "n/a", "na", "unknown",
             "prefiero no decir", "prefiero no indicar", "-", ""):
        return "No indica"
    return np.nan

df_in["gender"] = df_in["gender"].apply(normalizar_genero)

# ------------------------------------------------------------
# 2.9 FILTRAR SOLO VIDA
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[2.9] 🎯 FILTRO: SOLO SEGUROS DE VIDA")
print("-" * 60)

antes = len(df_in)
df_in = df_in[df_in["insurance_type"] == "life"].copy()
print(f"📉 Filas eliminadas: {antes - len(df_in)}  →  Restantes: {len(df_in)}")

# ------------------------------------------------------------
# 3. BUGS CRÍTICOS → dropna
# ------------------------------------------------------------
antes = len(df_in)
df_in = df_in.dropna(subset=[
    "policy_id", "customer_id", "premium_amount_eur", "policy_start_date",
])
print(f"\n[3] 🫀 Filas eliminadas por falta de datos críticos: {antes - len(df_in)}")
print(f"    ✅ Filas restantes: {len(df_in)}")

# ------------------------------------------------------------
# 4. NULOS LÓGICOS INDEPENDIENTES  → SE RELLENAN CON CENTINELAS
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[4] 🤔 TRATAMIENTO DE NULOS LÓGICOS INDEPENDIENTES")
print("-" * 60)

rellenos_independientes = {
    "customer_phone":  "Sin Teléfono",
    "complaint_type":  "Sin Queja",
    "broker_id":       "Sin Broker",
    "notes":           "Sin Notas",
    "gender":          "No indica",
}
for col, valor in rellenos_independientes.items():
    n = df_in[col].isnull().sum()
    df_in[col] = df_in[col].fillna(valor)
    print(f"    ✅ {col:<28} → '{valor}'  ({n} filas)")

# Montos neutros y flags
n = df_in["outstanding_balance_eur"].isnull().sum()
df_in["outstanding_balance_eur"] = df_in["outstanding_balance_eur"].fillna(0)
print(f"    ✅ {'outstanding_balance_eur':<28} → 0  ({n} filas)")

n = df_in["fraud_flag"].isnull().sum()
df_in["fraud_flag"] = df_in["fraud_flag"].fillna(False).astype(bool)
print(f"    ✅ {'fraud_flag':<28} → False  ({n} filas)")

# Encuestas: nulo = "no respondió" → 0 (numérico, no texto)
for col in ["customer_satisfaction_score", "nps_score"]:
    n = df_in[col].isnull().sum()
    df_in[col] = pd.to_numeric(df_in[col], errors="coerce").fillna(0)
    print(f"    ✅ {col:<28} → 0 (no respondió)  ({n} filas)")

# ------------------------------------------------------------
# 5. DROP COLUMNAS VEHÍCULO/PROPIEDAD
# ------------------------------------------------------------
cols_no_aplican_vida = [
    "vehicle_make", "vehicle_model", "vehicle_year", "vehicle_value_eur",
    "property_type", "property_value_eur", "property_year_built",
]
cols_presentes = [c for c in cols_no_aplican_vida if c in df_in.columns]
if cols_presentes:
    df_in = df_in.drop(columns=cols_presentes)
    print(f"\n[5] ✂️ Columnas eliminadas (no aplican a Vida): {len(cols_presentes)}")
    for c in cols_presentes:
        print(f"    ✗ {c}")

# ------------------------------------------------------------
# 6. CLAIM: clasificar pólizas según siniestro
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[6] 💡 CLASIFICAR PÓLIZAS SEGÚN SINIESTRO")
print("-" * 60)

sin_claim = df_in["claim_id"].isna()
con_claim = ~sin_claim
print(f"    Pólizas SIN siniestro: {sin_claim.sum():,}")
print(f"    Pólizas CON siniestro: {con_claim.sum():,}")

# Rellenamos con 0 los montos de las pólizas sin siniestro.
df_in.loc[sin_claim, "claim_amount_eur"]          = df_in.loc[sin_claim, "claim_amount_eur"].fillna(0)
df_in.loc[sin_claim, "claim_approved_amount_eur"] = df_in.loc[sin_claim, "claim_approved_amount_eur"].fillna(0)
df_in.loc[sin_claim, "settlement_amount_eur"]     = df_in.loc[sin_claim, "settlement_amount_eur"].fillna(0)

# Bug crítico: siniestro SÍ reportado pero SIN settlement → eliminar
mask_bug = con_claim & df_in["settlement_amount_eur"].isna()
print(f"    🫀 Filas eliminadas (siniestro sin settlement): {mask_bug.sum()}")
df_in = df_in[~mask_bug]

# ------------------------------------------------------------
# 6.5 NULOS LÓGICOS DE RECLAMO  → TODOS SE RELLENAN
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[6.5] 🔧 NULOS LÓGICOS DE RECLAMO")
print("-" * 60)

# Recalculamos máscaras tras el filtro
sin_claim = df_in["claim_id"].isna()
con_claim = ~sin_claim

# --- SIN siniestro: etiquetas "Sin X" ---
df_in.loc[sin_claim, "claim_id"]             = "Sin Siniestro"
df_in.loc[sin_claim, "claim_description"]    = "Sin Descripción"
df_in.loc[sin_claim, "adjuster_id"]          = "Sin Asignar"
df_in.loc[sin_claim, "adjuster_name"]        = "Sin Asignar"
df_in.loc[sin_claim, "claim_denial_reason"]  = "No Aplica"
df_in.loc[sin_claim, "police_report_number"] = "Sin Reporte Policial"
print(f"    ✅ Sin siniestro ({sin_claim.sum():,} filas) → etiquetas 'Sin X'")

# --- CON siniestro: rellenamos TODOS los nulos con centinela ---
n_adj_id   = df_in.loc[con_claim, "adjuster_id"].isnull().sum()
n_adj_name = df_in.loc[con_claim, "adjuster_name"].isnull().sum()
n_desc     = df_in.loc[con_claim, "claim_description"].isnull().sum()
n_denial   = df_in.loc[con_claim, "claim_denial_reason"].isnull().sum()
n_police   = df_in.loc[con_claim, "police_report_number"].isnull().sum()
n_claim_id = df_in.loc[con_claim, "claim_id"].isnull().sum()

df_in.loc[con_claim, "adjuster_id"]          = df_in.loc[con_claim, "adjuster_id"].fillna("Ajustador Desconocido")
df_in.loc[con_claim, "adjuster_name"]        = df_in.loc[con_claim, "adjuster_name"].fillna("Ajustador Desconocido")
df_in.loc[con_claim, "claim_description"]    = df_in.loc[con_claim, "claim_description"].fillna("Descripción Faltante")
df_in.loc[con_claim, "claim_denial_reason"]  = df_in.loc[con_claim, "claim_denial_reason"].fillna("No Especificado")
df_in.loc[con_claim, "police_report_number"] = df_in.loc[con_claim, "police_report_number"].fillna("Sin Reporte Policial")
df_in.loc[con_claim, "claim_id"]             = df_in.loc[con_claim, "claim_id"].fillna("Siniestro Desconocido")

print(f"    ⚠️  Con siniestro ({con_claim.sum():,} filas) → nulos marcados como bugs:")
print(f"        claim_id:             {n_claim_id:,} → 'Siniestro Desconocido'")
print(f"        adjuster_id:          {n_adj_id:,} → 'Ajustador Desconocido'")
print(f"        adjuster_name:        {n_adj_name:,} → 'Ajustador Desconocido'")
print(f"        claim_description:    {n_desc:,} → 'Descripción Faltante'")
print(f"        claim_denial_reason:  {n_denial:,} → 'No Especificado'")
print(f"        police_report_number: {n_police:,} → 'Sin Reporte Policial'")

# ------------------------------------------------------------
# 6.9 RELLENO GENÉRICO DE SEGURIDAD
# ------------------------------------------------------------
# 📌 Cualquier columna que aún tenga nulos (por ejemplo, fechas de
#    siniestro, montos numéricos de claim, campos varios) se rellena
#    para garantizar que el CSV final salga SIN nulos.
print("\n" + "-" * 60)
print("[6.9] 🛟 RELLENO GENÉRICO DE SEGURIDAD (para 0 nulos)")
print("-" * 60)

centinelas_texto = {
    "claim_date": "Sin Fecha",
    "incident_date": "Sin Fecha",
    "incident_location": "Desconocido",
    "adjustment_date": "Sin Fecha",
    "settlement_date": "Sin Fecha",
    "customer_address": "Desconocido",
    "customer_city": "Desconocido",
    "customer_state": "Desconocido",
    "customer_country": "Desconocido",
}

# 1) Rellenar columnas de texto con centinelas específicos si están definidos
for col, valor in centinelas_texto.items():
    if col in df_in.columns:
        n = df_in[col].isnull().sum()
        if n > 0:
            df_in[col] = df_in[col].fillna(valor)
            print(f"    ✅ {col:<28} → '{valor}'  ({n} filas)")

# 2) Rellenar cualquier columna object restante con "Desconocido"
cols_object = df_in.select_dtypes(include=["object"]).columns
for col in cols_object:
    n = df_in[col].isnull().sum()
    if n > 0:
        df_in[col] = df_in[col].fillna("Desconocido")
        print(f"    ✅ {col:<28} → 'Desconocido'  ({n} filas)")

# 3) Rellenar cualquier columna numérica restante con 0
cols_num = df_in.select_dtypes(include=[np.number]).columns
for col in cols_num:
    n = df_in[col].isnull().sum()
    if n > 0:
        df_in[col] = df_in[col].fillna(0)
        print(f"    ✅ {col:<28} → 0  ({n} filas)")

# ------------------------------------------------------------
# 7. EDAD: mediana por género
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[7] 📊 EDAD: mediana por género")
print("-" * 60)

df_in["age"] = pd.to_numeric(df_in["age"], errors="coerce")
antes_nulos = df_in["age"].isna().sum()
print(f"    Edades nulas antes: {antes_nulos}")

mediana_por_genero = df_in.groupby("gender", dropna=False)["age"].median()
print(f"    Mediana por género: {mediana_por_genero.to_dict()}")

df_in["age"] = df_in.groupby("gender", dropna=False)["age"].transform(
    lambda s: s.fillna(s.median())
)
df_in["age"] = df_in["age"].fillna(df_in["age"].median())
print(f"    ✅ Edades nulas después: {df_in['age'].isna().sum()}")

# ------------------------------------------------------------
# 7.5 MAPEO DE CÓDIGOS POSTALES FICTICIOS → REALES
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[7.5] 📮 MAPEO DE CÓDIGOS POSTALES")
print("-" * 60)

def mapear_codigos_postales_reales(
    df_seguros, df_germany,
    col_city_seguros="city", col_state_seguros="state",
    col_city_germany="municipality_name_(gemeindename)",
    col_state_germany="state_(land)",
    col_postal_germany="postal_code",
):
    df = df_seguros.copy()
    df["postal_code_original"] = df["postal_code"]

    def _norm(s):
        if pd.isna(s):
            return np.nan
        s = str(s).strip().lower()
        for k, v in {"ü":"u","ö":"o","ä":"a","ß":"ss","é":"e","è":"e","á":"a","à":"a"}.items():
            s = s.replace(k, v)
        return s

    df_ger = df_germany.copy()
    df_ger["_city_norm"]  = df_ger[col_city_germany].apply(_norm)
    df_ger["_state_norm"] = df_ger[col_state_germany].apply(_norm)
    df_ger["_cp"] = df_ger[col_postal_germany].astype(str).str.strip()

    df_ger_unicos = df_ger.dropna(subset=["_state_norm", "_city_norm", "_cp"])
    df_ger_unicos = df_ger_unicos.drop_duplicates(subset=["_state_norm", "_city_norm"])
    mapa_estado_ciudad = df_ger_unicos.set_index(["_state_norm", "_city_norm"])["_cp"].to_dict()

    df_ger_solo_ciudad = df_ger.dropna(subset=["_city_norm", "_cp"])
    df_ger_solo_ciudad = df_ger_solo_ciudad.drop_duplicates(subset=["_city_norm"])
    mapa_ciudad = df_ger_solo_ciudad.set_index("_city_norm")["_cp"].to_dict()
    ciudades_alemania = list(mapa_ciudad.keys())

    def _asignar_cp(row):
        ciudad = _norm(row.get(col_city_seguros))
        estado = _norm(row.get(col_state_seguros))
        if pd.isna(ciudad):
            return np.nan
        if (estado, ciudad) in mapa_estado_ciudad:
            return mapa_estado_ciudad[(estado, ciudad)]
        if ciudad in mapa_ciudad:
            return mapa_ciudad[ciudad]
        for ciudad_ger in ciudades_alemania:
            if ciudad in ciudad_ger or ciudad_ger in ciudad:
                return mapa_ciudad[ciudad_ger]
        alias = {
            "munich":"munchen","cologne":"koln","hanover":"hannover",
            "nuremberg":"nurnberg","vienna":"wien","brunswick":"braunschweig",
            "mayence":"mainz",
        }
        ciudad_alias = alias.get(ciudad, ciudad)
        if ciudad_alias in mapa_ciudad:
            return mapa_ciudad[ciudad_alias]
        return np.nan

    df["postal_code"] = df.apply(_asignar_cp, axis=1)

    total = len(df)
    mapeados = df["postal_code"].notna().sum()
    print(f"    📊 Total: {total:,}  |  Mapeados: {mapeados:,} ({mapeados/total:.1%})")
    return df

df_in = mapear_codigos_postales_reales(df_in, df_germany)

# Si aún quedan nulos en postal_code, rellenar con el original
n_cp = df_in["postal_code"].isnull().sum()
if n_cp > 0:
    df_in["postal_code"] = df_in["postal_code"].fillna(df_in["postal_code_original"])
    df_in["postal_code"] = df_in["postal_code"].fillna("Desconocido")
    print(f"    ✅ postal_code: {n_cp} nulos rellenados con original o 'Desconocido'")

# ------------------------------------------------------------
# 8. AUDITORÍA FINAL
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("🧾 AUDITORÍA FINAL")
print("-" * 60)

print(f"Filas finales: {len(df_in):,}")
print(f"Columnas finales: {df_in.shape[1]}")
print(f"\n🔑 Duplicados en policy_id: {df_in.duplicated(subset=['policy_id']).sum()}")

print(f"\n🔍 Nulos restantes (solo columnas con nulos):")
nulos = df_in.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
if len(nulos) == 0:
    print("    ✅ Sin nulos restantes")
else:
    print(nulos.to_string())

# ------------------------------------------------------------
# 9. GUARDAR
# ------------------------------------------------------------
df_in.to_csv("insurance_life_only_clean.csv", index=False)
print(f"\n✅ Guardado: insurance_life_only_clean.csv ({len(df_in)} filas)")

------------------------------------------------------------
INICIO LIMPIEZA: insurance_policies_claims
------------------------------------------------------------

[1] 📥 Cargado: 52676 filas x 76 columnas
    Alemania cargada: 10950 filas

[2] 🏷️ Columnas ya estandarizadas (no se requiere rename)

[2.1] ✂️ Columnas eliminadas: ['customer_email']

------------------------------------------------------------
[2.5] 🩺 DIAGNÓSTICO DE NULOS Y VACÍOS
------------------------------------------------------------
                            tipo_dato  vacios_str  nan_reales  total_problemas  pct_del_total
property_type                  object       31992           0            31992          60.73
broker_id                      object       30825           0            30825          58.52
property_year_built            object       27281           0            27281          51.79
nps_score                      object       25252           0            25252          47.94
claim_id           

In [15]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# ENRIQUECIMIENTO — Seguros + Municipios (merge sin duplicados)
# ------------------------------------------------------------
print("=" * 60)
print("🔗 ENRIQUECIMIENTO: Seguros + Alemania")
print("=" * 60)

# ------------------------------------------------------------
# 1. CARGA
# ------------------------------------------------------------
df_seguros = pd.read_csv("insurance_life_only_clean.csv", dtype=str, keep_default_na=False)
df_alemania = pd.read_csv("germany_municipalities_clean.csv", dtype=str, keep_default_na=False)

print(f"📥 Seguros:  {df_seguros.shape[0]:,} filas x {df_seguros.shape[1]} columnas")
print(f"📥 Alemania: {df_alemania.shape[0]:,} filas x {df_alemania.shape[1]} columnas")

# ------------------------------------------------------------
# 2. NORMALIZAR LLAVE (postal_code)
# ------------------------------------------------------------
def normalizar_cp(s):
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"[^\d]", "", regex=True)
        .replace({"": np.nan, "nan": np.nan})
        .str.zfill(5)
    )

df_seguros["postal_code"] = normalizar_cp(df_seguros["postal_code"])
df_alemania["postal_code"] = normalizar_cp(df_alemania["postal_code"])

# ------------------------------------------------------------
# 3. RENOMBRAR SEGUROS
# ------------------------------------------------------------
rename_seguros = {
    "city":    "customer_city",
    "address": "customer_address",
    "state":   "customer_state",
    "country": "customer_country",
}
rename_seguros = {k: v for k, v in rename_seguros.items() if k in df_seguros.columns}
df_seguros = df_seguros.rename(columns=rename_seguros)

# ------------------------------------------------------------
# 4. RENOMBRAR ALEMANIA
# ------------------------------------------------------------
rename_alemania = {
    "state":              "municipality_state",
    "district":           "municipality_district",
    "regional_type":      "municipality_regional_type",
    "municipality_name":  "municipality_name",
    "population_total":   "municipality_population",
    "population_density": "municipality_density",
    "male":               "municipality_male",
    "female":             "municipality_female",
    "latitude":           "municipality_latitude",
    "longitude":          "municipality_longitude",
}
rename_alemania = {k: v for k, v in rename_alemania.items() if k in df_alemania.columns}
df_alemania = df_alemania.rename(columns=rename_alemania)

# ------------------------------------------------------------
# 5. CONVERTIR NUMÉRICOS EN ALEMANIA
# ------------------------------------------------------------
cols_numericas = [
    "municipality_population", "municipality_density",
    "municipality_male", "municipality_female",
    "municipality_latitude", "municipality_longitude",
]
cols_numericas = [c for c in cols_numericas if c in df_alemania.columns]
for col in cols_numericas:
    df_alemania[col] = pd.to_numeric(
        df_alemania[col].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )

# ------------------------------------------------------------
# 6. AGREGAR ALEMANIA A 1 FILA POR CP  ⚠️ CLAVE PARA NO DUPLICAR
# ------------------------------------------------------------
# 📌 Alemania tiene varios municipios por CP. Si hacemos merge
#    directo, cada póliza se multiplica. Agregamos a 1 fila por
#    CP antes del merge para evitar el efecto "147% match".
print("\n🧮 Agregando Alemania a 1 fila por CP...")

agg_dict = {}
for col in ["municipality_name", "municipality_state",
            "municipality_district", "municipality_regional_type"]:
    if col in df_alemania.columns:
        agg_dict[col] = "first"
for col in ["municipality_population", "municipality_male", "municipality_female"]:
    if col in df_alemania.columns:
        agg_dict[col] = "sum"
for col in ["municipality_density", "municipality_latitude", "municipality_longitude"]:
    if col in df_alemania.columns:
        agg_dict[col] = "mean"

df_alemania_unico = df_alemania.groupby("postal_code", as_index=False).agg(agg_dict)
print(f"    Alemania: {len(df_alemania):,} → {len(df_alemania_unico):,} filas (1 por CP)")

# ------------------------------------------------------------
# 7. MERGE
# ------------------------------------------------------------
print("\n🔗 Haciendo merge...")
df_final = pd.merge(
    df_seguros,
    df_alemania_unico,
    on="postal_code",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print(f"\n🚦 Auditoría del match:")
print(df_final["_merge"].value_counts().to_string())

match_counts = df_final["_merge"].value_counts()
total = len(df_final)
if "both" in match_counts:
    pct = match_counts["both"] / total * 100
    print(f"\n  ✅ Match exitoso: {match_counts['both']:,} filas ({pct:.2f}%)")
if "left_only" in match_counts:
    pct = match_counts["left_only"] / total * 100
    print(f"  ⚠️  Sin match: {match_counts['left_only']:,} filas ({pct:.2f}%)")

# ------------------------------------------------------------
# 8. RELLENAR NULOS INTRODUCIDOS POR EL MERGE
# ------------------------------------------------------------
# 📌 Las pólizas cuyo CP no existe en el catálogo alemán quedan
#    con nulos en las columnas de municipio. Las marcamos como
#    "Desconocido" para que no queden vacías.
print("\n🔧 Rellenando nulos introducidos por el merge...")
cols_municipio = [c for c in df_final.columns if c.startswith("municipality_")]
for col in cols_municipio:
    n = df_final[col].isnull().sum()
    if n > 0:
        if df_final[col].dtype == "object":
            df_final[col] = df_final[col].fillna("Desconocido")
        else:
            df_final[col] = df_final[col].fillna(0)
        print(f"    ✅ {col:<30} → {n} nulos rellenados")

# Quitar columna _merge
df_final = df_final.drop(columns=["_merge"])

# ------------------------------------------------------------
# 9. GUARDAR
# ------------------------------------------------------------
df_final.to_csv("insurance_life_enriched_v1.csv", index=False)
print(f"\n✅ Guardado: insurance_life_enriched_v1.csv ({len(df_final):,} filas)")

# ------------------------------------------------------------
# 10. AUDITORÍA FINAL
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("🧾 AUDITORÍA FINAL DEL ENRIQUECIDO")
print("=" * 60)
print(f"Filas: {len(df_final):,}")
print(f"Columnas: {df_final.shape[1]}")
print(f"Duplicados en policy_id: {df_final.duplicated(subset=['policy_id']).sum()}")

nulos = df_final.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
if len(nulos) == 0:
    print("\n✅ Sin nulos restantes")
else:
    print(f"\n⚠️  Nulos restantes en {len(nulos)} columnas:")
    print(nulos.to_string())

🔗 ENRIQUECIMIENTO: Seguros + Alemania
📥 Seguros:  6,593 filas x 69 columnas
📥 Alemania: 10,757 filas x 12 columnas

🧮 Agregando Alemania a 1 fila por CP...
    Alemania: 10,757 → 6,440 filas (1 por CP)

🔗 Haciendo merge...

🚦 Auditoría del match:
_merge
both          6593
left_only        0
right_only       0

  ✅ Match exitoso: 6,593 filas (100.00%)
  ⚠️  Sin match: 0 filas (0.00%)

🔧 Rellenando nulos introducidos por el merge...

✅ Guardado: insurance_life_enriched_v1.csv (6,593 filas)

🧾 AUDITORÍA FINAL DEL ENRIQUECIDO
Filas: 6,593
Columnas: 79
Duplicados en policy_id: 81

✅ Sin nulos restantes


In [19]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# ENRIQUECIMIENTO v3 — Seguros + Alemania + Estados + groupby
# Garantiza CERO nulos al final
# ------------------------------------------------------------
print("=" * 60)
print("🔗 ENRIQUECIMIENTO v3: Seguros + Alemania + Estados + groupby")
print("=" * 60)

# ------------------------------------------------------------
# 1. CARGA
# ------------------------------------------------------------
df_seguros = pd.read_csv("insurance_life_only_clean.csv", dtype=str, keep_default_na=False)
df_alemania = pd.read_csv("germany_municipalities_clean.csv", dtype=str, keep_default_na=False)

print(f"📥 Seguros:  {df_seguros.shape[0]:,} x {df_seguros.shape[1]}")
print(f"📥 Alemania: {df_alemania.shape[0]:,} x {df_alemania.shape[1]}")

# ------------------------------------------------------------
# 2. NORMALIZAR CP
# ------------------------------------------------------------
def normalizar_cp(s):
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"[^\d]", "", regex=True)
        .replace({"": np.nan, "nan": np.nan})
        .str.zfill(5)
    )

df_seguros["postal_code"] = normalizar_cp(df_seguros["postal_code"])
df_alemania["postal_code"] = normalizar_cp(df_alemania["postal_code"])

# ------------------------------------------------------------
# 3. RENOMBRAR
# ------------------------------------------------------------
df_seguros = df_seguros.rename(columns={
    "city": "customer_city", "address": "customer_address",
    "state": "customer_state", "country": "customer_country",
})

df_alemania = df_alemania.rename(columns={
    "state":              "municipality_state",
    "district":           "municipality_district",
    "regional_type":      "municipality_regional_type",
    "municipality_name":  "municipality_name",
    "population_total":   "municipality_population",
    "population_density": "municipality_density",
    "male":               "municipality_male",
    "female":             "municipality_female",
    "latitude":           "municipality_latitude",
    "longitude":          "municipality_longitude",
})

# ------------------------------------------------------------
# 4. NORMALIZAR ESTADOS DEL CLIENTE
# ------------------------------------------------------------
mapeo_estados = {
    "bavaria":"bayern","saxony":"sachsen","hesse":"hessen",
    "north rhine-westphalia":"nordrhein-westfalen","nrw":"nordrhein-westfalen",
    "lower saxony":"niedersachsen","rhineland-palatinate":"rheinland-pfalz",
    "thuringia":"thüringen","saxony-anhalt":"sachsen-anhalt",
    "mecklenburg-western pomerania":"mecklenburg-vorpommern",
    "baden-wuerttemberg":"baden-württemberg","baden-wurttemberg":"baden-württemberg",
    "bw":"baden-württemberg","by":"bayern","be":"berlin","bb":"brandenburg",
    "hb":"bremen","hh":"hamburg","he":"hessen","mv":"mecklenburg-vorpommern",
    "ni":"niedersachsen","nw":"nordrhein-westfalen","rp":"rheinland-pfalz",
    "sl":"saarland","sn":"sachsen","st":"sachsen-anhalt",
    "sh":"schleswig-holstein","th":"thüringen",
    "bayern":"bayern","sachsen":"sachsen","hessen":"hessen",
    "nordrhein-westfalen":"nordrhein-westfalen","niedersachsen":"niedersachsen",
    "baden-württemberg":"baden-württemberg","rheinland-pfalz":"rheinland-pfalz",
    "schleswig-holstein":"schleswig-holstein","brandenburg":"brandenburg",
    "berlin":"berlin","hamburg":"hamburg","bremen":"bremen",
    "thüringen":"thüringen","thuringen":"thüringen",
    "sachsen-anhalt":"sachsen-anhalt","mecklenburg-vorpommern":"mecklenburg-vorpommern",
    "saarland":"saarland",
    "n/a":None,"na":None,"-":None,"":None,
}

df_seguros["customer_state_norm"] = (
    df_seguros["customer_state"].str.lower().str.strip().map(mapeo_estados)
)

# ------------------------------------------------------------
# 5. CONVERTIR NUMÉRICOS EN ALEMANIA
# ------------------------------------------------------------
for col in ["municipality_population", "municipality_density",
            "municipality_male", "municipality_female",
            "municipality_latitude", "municipality_longitude"]:
    df_alemania[col] = pd.to_numeric(
        df_alemania[col].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )

# ------------------------------------------------------------
# 6. AGREGAR ALEMANIA A 1 FILA POR CP
# ------------------------------------------------------------
df_alemania_cp = df_alemania.groupby("postal_code", as_index=False).agg({
    "municipality_name":       "first",
    "municipality_state":      "first",
    "municipality_district":   "first",
    "municipality_population": "sum",
    "municipality_density":    "mean",
    "municipality_latitude":   "mean",
    "municipality_longitude":  "mean",
})

print(f"\n🧮 Alemania por CP: {len(df_alemania):,} → {len(df_alemania_cp):,} filas")

# ------------------------------------------------------------
# 7. MERGE POR CP
# ------------------------------------------------------------
df_seguros = pd.merge(
    df_seguros, df_alemania_cp,
    on="postal_code", how="left",
    validate="many_to_one",
    suffixes=("", "_cp"),
)

match_cp = df_seguros["municipality_name"].notna().sum()
pct_cp = match_cp / len(df_seguros) * 100
print(f"\n✅ Merge por CP: {match_cp:,} / {len(df_seguros):,} ({pct_cp:.1f}%)")

# ------------------------------------------------------------
# 8. MERGE POR ESTADO (plan B para CP sin match)
# ------------------------------------------------------------
df_alemania_estado = df_alemania.groupby("municipality_state", as_index=False).agg({
    "municipality_population": "sum",
    "municipality_density":    "mean",
})

df_seguros = pd.merge(
    df_seguros,
    df_alemania_estado.rename(columns={
        "municipality_state":      "state_key",
        "municipality_population": "state_population",
        "municipality_density":    "state_density",
    }),
    left_on="customer_state_norm", right_on="state_key",
    how="left", validate="many_to_one",
)

match_estado = df_seguros["state_key"].notna().sum()
pct_estado = match_estado / len(df_seguros) * 100
print(f"✅ Merge por estado: {match_estado:,} / {len(df_seguros):,} ({pct_estado:.1f}%)")

# ------------------------------------------------------------
# 9. AGREGAR POR policy_id (1 fila por póliza)
# ------------------------------------------------------------
cols_agg = {}
for c in df_seguros.columns:
    if c == "policy_id":
        continue
    if c in ("claim_amount_eur", "claim_approved_amount_eur", "settlement_amount_eur"):
        cols_agg[c] = "sum"
    elif c in ("fraud_score", "risk_score"):
        cols_agg[c] = "max"
    else:
        cols_agg[c] = "first"

df_seguros["_n_claims"] = 1
cols_agg["_n_claims"] = "sum"

df_final = df_seguros.groupby("policy_id", as_index=False).agg(cols_agg)
df_final = df_final.rename(columns={"_n_claims": "num_claims_detail"})

print(f"\n🔽 Agrupado: {len(df_seguros):,} → {len(df_final):,} filas (1 por policy_id)")

# ------------------------------------------------------------
# 10. COMBINAR municipio ↔ estado
# ------------------------------------------------------------
df_final["municipality_population"] = df_final["municipality_population"].fillna(
    df_final["state_population"]
)
df_final["municipality_density"] = df_final["municipality_density"].fillna(
    df_final["state_density"]
)
df_final = df_final.drop(columns=["state_population", "state_density", "state_key"])

# ------------------------------------------------------------
# 11. RELLENO DE NULOS INTRODUCIDOS POR MERGES (TODOS)
# ------------------------------------------------------------
print("\n🔧 Rellenando nulos introducidos por los merges...")

# 11.1 Columnas geográficas (texto)
cols_geo_texto = [
    "municipality_name", "municipality_state", "municipality_district",
    "municipality_regional_type",
]
for col in cols_geo_texto:
    if col in df_final.columns:
        n = df_final[col].isnull().sum()
        if n > 0:
            df_final[col] = df_final[col].fillna("Desconocido")
            print(f"    ✅ {col:<30} → 'Desconocido'  ({n} filas)")

# 11.2 Columnas geográficas (numéricas)
cols_geo_num = [
    "municipality_population", "municipality_density",
    "municipality_male", "municipality_female",
    "municipality_latitude", "municipality_longitude",
]
for col in cols_geo_num:
    if col in df_final.columns:
        n = df_final[col].isnull().sum()
        if n > 0:
            df_final[col] = df_final[col].fillna(0)
            print(f"    ✅ {col:<30} → 0  ({n} filas)")

# 11.3 Cualquier otra columna object que aún tenga nulos
for col in df_final.select_dtypes(include=["object"]).columns:
    n = df_final[col].isnull().sum()
    if n > 0:
        df_final[col] = df_final[col].fillna("Desconocido")
        print(f"    ✅ {col:<30} → 'Desconocido'  ({n} filas)")

# 11.4 Cualquier otra columna numérica que aún tenga nulos
for col in df_final.select_dtypes(include=[np.number]).columns:
    n = df_final[col].isnull().sum()
    if n > 0:
        df_final[col] = df_final[col].fillna(0)
        print(f"    ✅ {col:<30} → 0  ({n} filas)")

# ------------------------------------------------------------
# 12. VERIFICACIÓN FINAL
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("🧾 VERIFICACIÓN FINAL")
print("=" * 60)
print(f"Filas: {len(df_final):,}")
print(f"policy_id únicos: {df_final['policy_id'].nunique():,}")
print(f"Duplicados: {df_final['policy_id'].duplicated().sum()}")

assert df_final["policy_id"].duplicated().sum() == 0, "❌ Duplicados"

nulos = df_final.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
if len(nulos) == 0:
    print("\n✅ Sin nulos restantes — 0 nulos en TODAS las columnas")
else:
    print(f"\n⚠️  Nulos restantes:")
    print(nulos.to_string())

# ------------------------------------------------------------
# 13. GUARDAR
# ------------------------------------------------------------
df_final.to_csv("insurance_life_enriched.csv", index=False)
print(f"\n✅ Guardado: insurance_life_enriched.csv ({len(df_final):,} filas)")

# ------------------------------------------------------------
# 14. DESCARGA (Colab)
# ------------------------------------------------------------
from google.colab import files
files.download("insurance_life_enriched.csv")

🔗 ENRIQUECIMIENTO v3: Seguros + Alemania + Estados + groupby
📥 Seguros:  6,593 x 69
📥 Alemania: 10,757 x 12

🧮 Alemania por CP: 10,757 → 6,440 filas

✅ Merge por CP: 6,593 / 6,593 (100.0%)
✅ Merge por estado: 6,593 / 6,593 (100.0%)

🔽 Agrupado: 6,593 → 6,512 filas (1 por policy_id)

🔧 Rellenando nulos introducidos por los merges...

🧾 VERIFICACIÓN FINAL
Filas: 6,512
policy_id únicos: 6,512
Duplicados: 0

✅ Sin nulos restantes — 0 nulos en TODAS las columnas

✅ Guardado: insurance_life_enriched.csv (6,512 filas)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>